# 02 — ModélisationPipeline complet de modélisation :1. Chargement et prétraitement2. Baseline3. Régression Logistique4. Random Forest5. XGBoost6. Comparaison des modèles> ⚠️ L'attribut `duration` est **exclu** des modèles (car inconnu avant la fin de l'appel).

In [ ]:
# Importsimport pandas as pdimport numpy as npimport syssys.path.insert(0, '..')from src.data_loader import load_bank, load_bank_additionalfrom src.preprocessing import (    separate_features_target,    get_column_types,    split_data,    build_preprocessing_pipeline,)from src.modeling import (    get_baseline,    get_logistic_regression,    get_random_forest,    get_xgboost,    tune_model,)from src.evaluation import (    get_metrics,    print_classification_report,    plot_confusion_matrix,    plot_roc_curve,    plot_feature_importance,    compare_models,)from src.utils import set_seed, Timer, save_modelset_seed(42)%matplotlib inline

## 1. Chargement des données

In [ ]:
# Choisir le dataset à utiliserdf = load_bank(full=True)          # Option 1 : Bank (UCI original)# df = load_bank_additional(full=True)  # Option 2 : Bank Additional (enrichi)print(f"Shape: {df.shape}")

## 2. Séparation features / target

In [ ]:
X, y = separate_features_target(df, drop_duration=True)print(f"X: {X.shape}, y: {y.shape}")print(f"Class distribution:\n{y.value_counts()}")

## 3. Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, stratify=True)

## 4. Prétraitement

In [ ]:
num_cols, cat_cols = get_column_types(X_train)print(f"Numerical columns ({len(num_cols)}): {num_cols}")print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)X_train_pp = preprocessor.fit_transform(X_train)X_test_pp = preprocessor.transform(X_test)print(f"X_train after preprocessing: {X_train_pp.shape}")

## 5. Modèle baseline

In [ ]:
baseline = get_baseline(strategy='stratified')baseline.fit(X_train_pp, y_train)y_pred_base = baseline.predict(X_test_pp)print("Baseline (stratified):")print(get_metrics(y_test, y_pred_base))

## 6. Régression Logistique

In [ ]:
lr = get_logistic_regression(class_weight='balanced', max_iter=2000)with Timer("Logistic Regression"):    lr.fit(X_train_pp, y_train)y_pred_lr = lr.predict(X_test_pp)y_proba_lr = lr.predict_proba(X_test_pp)[:, 1]print_classification_report(y_test, y_pred_lr)

In [ ]:
plot_confusion_matrix(y_test, y_pred_lr, title='Logistic Regression')plot_roc_curve(y_test, y_proba_lr, label='Logistic Regression')

## 7. Random Forest

In [ ]:
rf = get_random_forest(n_estimators=200, class_weight='balanced')with Timer("Random Forest"):    rf.fit(X_train_pp, y_train)y_pred_rf = rf.predict(X_test_pp)y_proba_rf = rf.predict_proba(X_test_pp)[:, 1]print_classification_report(y_test, y_pred_rf)

In [ ]:
plot_confusion_matrix(y_test, y_pred_rf, title='Random Forest')plot_roc_curve(y_test, y_proba_rf, label='Random Forest')plot_feature_importance(rf, num_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)))

## 8. XGBoost

In [ ]:
neg, pos = np.bincount(y_train)scale_pos_weight = neg / posxgb = get_xgboost(scale_pos_weight=scale_pos_weight, n_estimators=200)with Timer("XGBoost"):    xgb.fit(X_train_pp, y_train)y_pred_xgb = xgb.predict(X_test_pp)y_proba_xgb = xgb.predict_proba(X_test_pp)[:, 1]print_classification_report(y_test, y_pred_xgb)

In [ ]:
plot_confusion_matrix(y_test, y_pred_xgb, title='XGBoost')plot_roc_curve(y_test, y_proba_xgb, label='XGBoost')plot_feature_importance(xgb, num_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols)))

## 9. Comparaison des modèles

In [ ]:
results = {    "Baseline": get_metrics(y_test, y_pred_base),    "Logistic Regression": get_metrics(y_test, y_pred_lr, y_proba_lr),    "Random Forest": get_metrics(y_test, y_pred_rf, y_proba_rf),    "XGBoost": get_metrics(y_test, y_pred_xgb, y_proba_xgb),}compare_models(results, sort_by="ROC-AUC")

## 10. Sauvegarde du meilleur modèle

In [ ]:
# save_model(rf, 'random_forest_bank')